In [1]:
# @title Imports
import collections
import numpy as np
import pandas as pd
from IPython import display
import scipy
import plotly.express as px
import plotly.graph_objects as go
import dataclasses
from google.colab import drive

In [2]:
def get_tables_from_url(url):
  print(url)
  return pd.read_html(url, storage_options={"User-Agent": "Mozilla/5.0"})

In [3]:
# https://serenesforest.net/binding-blade/characters/base-stats/
# https://fireemblemwiki.org/wiki/Name_chart/Fire_Emblem:_The_Binding_Blade
FE6_NOA_BY_SERENESFOREST_NAME = {
    'Allen': 'Alen',
    'Astohl': 'Astolfo',
    'Barth': 'Barthe',
    'Brunya': 'Brunnya',
    'Ellen': 'Elen',
    'Elphin': 'Elffin',
    'Fa': 'Fae',
    'Gale': 'Galle',
    'Gonzales': 'Gonzalez',
    'Guinevere': 'Guinivere',
    'Lalum': 'Larum',
    'Lott': 'Lot',
    'Miredy': 'Melady',
    'Murdoch': 'Murdock',
    'Narshen': 'Narcian',
    'Oujay': 'Ogier',
    'Percival': 'Perceval',
    'Ray': 'Raigh',
    'Shin': 'Sin',
    'Sofiya': 'Sophia',
    'Tate': 'Thea',
    'Thany': 'Shanna',
    'Treck': 'Trec',
    'Wendy': 'Gwendolyn',
    'Yodel': 'Yoder',
    'Zealot': 'Zelot',
    'Zeis': 'Zeiss'
}

FEMALE_CHAR_NAMES = [
    # FE7
    'Lyn',
    'Florina',
    'Serra',
    'Rebecca',
    'Priscilla',
    'Fiora',
    'Ninian',
    'Isadora',
    'Farina',
    'Louise',
    'Nino',
    'Vaida',
    'Karla',

    # FE6
    'Elen',
    'Shanna',
    'Clarine',
    'Dorothy',
    'Sue',
    'Lilina',
    'Gwendolyn',
    'Fir',
    'Thea',
    'Larum',
    'Echidna',
    'Cath',
    'Melady',
    'Cecilia',
    'Sophia',
    'Igrene',
    'Fae',
    'Niime',
    'Juno',
    'Brunnya',
    'Guinivere',

    # FE8
    'Eirika',
    'Vanessa',
    'Neimi',
    'Lute',
    'Natasha',
    'Tana',
    'Amelia',
    'Tethys',
    'Marisa',
    "L'Arachel",
    'Myrrh',
    'Syrene',
    'Ismaire',
    'Selena',
]

FEMALE_LOCKED_CLASSES = [
    'Blade Lord',
    'Cleric',
    'Dancer',
    'Falcoknight',
    'Lord (Lyn)',
    'Pegasus Knight',
    'Troubadour',
    'Valkyrie',
    'Recruit (all)',
    'Recruit',
    'Recruit (2)',
    'Recruit (3)',
    'Lord (Eirika)',
    'Great Lord (Eirika)'
]

# The "class promotion gains" table for FE7 does not list the pre-promoted
# version of classes.
FE7_PROMOTED_CLASS_BY_CLASS = {
    'Archer': 'Sniper',
    'Cavalier': 'Paladin',
    'Cleric': 'Bishop',
    'Fighter': 'Warrior',
    'Knight': 'General',
    'Lord (Eliwood)': 'Knight Lord',
    'Lord (Hector)': 'Great Lord',
    'Lord (Lyn)': 'Blade Lord',
    'Mage': 'Sage',
    'Mercenary': 'Hero',
    'Monk': 'Bishop',
    'Myrmidon': 'Swordmaster',
    'Nomad': 'Nomadic Trooper',
    'Pegasus Knight': 'Falcoknight',
    'Pirate': 'Berserker',
    'Shaman': 'Druid',
    'Thief': 'Assassin',
    'Transporter (Tent)': 'Transporter (Wagon)',
    'Troubadour': 'Valkyrie',
    'Wyvern Rider': 'Wyvern Lord'
}

FE8_MOUNTED_NON_PROMOTED_CLASSES = [
    'Cavalier',
    'Wyvern Rider',
    'Pegasus Knight',
    'Troubador'
]

# https://fe6.triangleattack.com/guides/hard-mode-bonuses
FE6_HM_BONUS_BY_CHAPTER = {
    chapter: (chapter + 1) // 2 + 3 for chapter in range(1, 25)
}


def get_url_by_table_name(game_title):
  return {
      'class_growth_rates': f'https://serenesforest.net/{game_title}/classes/growth-rates/',
      'class_max_stats': f'https://serenesforest.net/{game_title}/classes/maximum-stats/',
      'class_promo_gains': f'https://serenesforest.net/{game_title}/classes/promotion-gains/',
      'char_base_stats': f'https://serenesforest.net/{game_title}/characters/base-stats/',
      'char_growth_rates': f'https://serenesforest.net/{game_title}/characters/growth-rates/'
  }


fe6_url_by_table_name = get_url_by_table_name('binding-blade')
fe7_url_by_table_name = get_url_by_table_name('blazing-sword')
fe8_url_by_table_name = get_url_by_table_name('the-sacred-stones')


In [4]:
# @title Helper functions for cleaning data

IGNORE_COLUMNS = ['Weapon Ranks', 'Weapon ranks',
                  'Weapon Rank', 'Affin', 'Weapon EXP']

################################
# Ambiguity on Knoll's base stats
# https://serenesforest.net/the-sacred-stones/characters/base-stats/
# alludes to Knoll getting 1 auto-level, and that the numbers in the table
# already account for this and round to the nearest integer.

# https://fea.fewiki.net/fea.php?character=knoll&game=8e
# shows the pre-auto-level base stats for Ephraim's route,
# but shows the randomness of the 1 auto-level for Eirika's route.

# https://fireemblemwiki.org/wiki/Knoll
# https://fe8.triangleattack.com/characters/knoll
# both show the result of the 1 auto-level rounded to the nearest integer

# Here, we follow fea.fewiki.net's approach of considering his starting level
# as 9 instead of 10, and use his pre-auto-level base stats.
KNOLL_BASE_STAT_BY_NAME = {
    'Lv': 9,
    'HP': 21,
    'S/M': 12,
    'Skl': 9,
    'Spd': 8,
    'Def': 2,
    'Res': 10,
    'Lck': 0,
}
################################


CLASS_RENAME_MAP = {
    'Valkyie': 'Valkyrie',
    'Valkyria': 'Valkyrie',
    'Mamkute': 'Manakete',
    'Mamkute (F)': 'Manakete (F)',
    'Mamkute (M)': 'Manakete (M)',
    'Falcon Knight': 'Falcoknight',
    'Nmd Trooper': 'Nomadic Trooper',
    'Nmd Trooper (F)': 'Nomadic Trooper (F)',
    'Nmd Trooper (M)': 'Nomadic Trooper (M)',
    'Nomad Trooper': 'Nomadic Trooper',
    'Nomad Trooper (F)': 'Nomadic Trooper (F)',
    'Nomad Trooper (M)': 'Nomadic Trooper (M)',
    # https://serenesforest.net/the-sacred-stones/characters/base-stats/
    # This table lists Gonzales's class as Bandit, but other tables use Brigand.
    'Bandit': 'Brigand'
}


# FE6 extra characters have base stats on serenesforest, but not growth rates.
# https://serenesforest.net/binding-blade/characters/base-stats/
# https://serenesforest.net/binding-blade/characters/growth-rates/
# Manually enter them from
# https://fireemblem.fandom.com/wiki/Brunnya#Growth_Rates
# or
# https://fireemblemwiki.org/wiki/Brunnya#Stats
FE6_EXTRA_CHARACTER_GROWTH_RATES = [
    {
        'Name': 'Narcian',
        'HP': 85,
        'S/M': 50,
        'Skl': 10,
        'Spd': 10,
        'Lck': 30,
        'Def': 10,
        'Res': 5
    },
    {
        'Name': 'Galle',
        'HP': 40,
        'S/M': 20,
        'Skl': 20,
        'Spd': 15,
        'Lck': 30,
        'Def': 30,
        'Res': 5
    },

    {
        'Name': 'Hector',
        'HP': 85,
        'S/M': 50,
        'Skl': 10,
        'Spd': 10,
        'Lck': 30,
        'Def': 10,
        'Res': 5
    },
    {
        'Name': 'Brunnya',
        'HP': 85,
        'S/M': 50,
        'Skl': 10,
        'Spd': 10,
        'Lck': 30,
        'Def': 10,
        'Res': 5
    },
    {
        'Name': 'Eliwood',
        'HP': 60,
        'S/M': 20,
        'Skl': 30,
        'Spd': 35,
        'Lck': 25,
        'Def': 15,
        'Res': 5
    },
    {
        'Name': 'Murdock',
        'HP': 85,
        'S/M': 50,
        'Skl': 10,
        'Spd': 10,
        'Lck': 30,
        'Def': 10,
        'Res': 5
    },
    {
        'Name': 'Zephiel',
        'HP': 85,
        'S/M': 50,
        'Skl': 10,
        'Spd': 10,
        'Lck': 30,
        'Def': 10,
        'Res': 5
    },
    {
        'Name': 'Guinivere',
        'HP': 60,
        'S/M': 20,
        'Skl': 30,
        'Spd': 35,
        'Lck': 25,
        'Def': 15,
        'Res': 5
    },
]


def remove_redundant_header_rows(df):
  """The header row is sometimes repeated as another row farther down the table
  to facilitate readability for tall tables. Remove these redundant rows."""
  c = df.columns[1]
  return df[df[c] != c].copy()


def append_char_gender(df):
  df = df.copy()
  df['Gender'] = df['Name'].apply(
      lambda n: 'F' if n in FEMALE_CHAR_NAMES else 'M')
  return df


def append_hm_bonus(df):
  df = df.copy()
  df['HM Bonus'] = 0

  ### FE7 ###
  # https://fe7.triangleattack.com/guides/hard-mode-bonuses
  # The following recruitable enemies from FE7 get a 5 level bonus
  # in Hector Hard Mode.
  fe7_hard_mode_bonus_chars = [
      'Guy', 'Raven', 'Legault', 'Heath', 'Geitz', 'Harken', 'Vaida'
  ]
  for char in fe7_hard_mode_bonus_chars:
    df.loc[df['Name'] == char, 'HM Bonus'] = 5

  ### FE6 ###
  # https://fe6.triangleattack.com/guides/hard-mode-bonuses
  # FE6 Hard Mode bonuses depend on the chapter recruited.

  # Percival only gets a HM bonus in Chapter 15.
  perceval_idx = (df['Name'] == 'Perceval')
  perceval_13 = df.loc[perceval_idx].copy().assign(Variant='Chapter 13')
  perceval_15 = df.loc[perceval_idx].copy().assign(
      Variant='Chapter 15').assign(**{'HM Bonus': FE6_HM_BONUS_BY_CHAPTER[15]})
  df = pd.concat([df.loc[~perceval_idx], perceval_13, perceval_15])

  # Cath can be recruited in different chapters, which changes the bonus.
  cath_idx = (df['Name'] == 'Cath')
  cath_rows = [
      df.loc[cath_idx].copy().assign(
          **{'Variant': f'Chapter {chapter}', 'HM Bonus': FE6_HM_BONUS_BY_CHAPTER[chapter]})
      for chapter in (12, 16, 20, 22)
  ]
  df = pd.concat([df.loc[~cath_idx], *cath_rows])

  # Klein/Thea are recruited in different chapters depending on the route.
  klein_thea_idx = (df['Name'].isin(['Klein', 'Thea']))
  klein_thea_rows = [
      df.loc[klein_thea_idx].copy().assign(
          **{
              'Variant': f"Chapter 11A (Larum's route)",
              'HM Bonus': FE6_HM_BONUS_BY_CHAPTER[11]
          }),
      df.loc[klein_thea_idx].copy().assign(
          **{
              'Variant': f"Chapter 10B Elffin's route)",
              'HM Bonus': FE6_HM_BONUS_BY_CHAPTER[10]
          })
  ]
  df = pd.concat([df.loc[~klein_thea_idx], *klein_thea_rows])

  # The remaining characters have a fixed chapter.
  hm_chapter_by_character = {
      'Rutger': 4,
      'Fir': 9,
      'Sin': 9,
      'Gonzalez': 10,
      'Melady': 13,
      'Garret': 15,
      'Zeiss': 16
  }
  for char, chapter in hm_chapter_by_character.items():
    df.loc[df['Name'] == char, 'HM Bonus'] = FE6_HM_BONUS_BY_CHAPTER[chapter]
  return df


def process_char_base_stats(df, is_fe7):
  df = df.copy()
  # https://serenesforest.net/the-sacred-stones/characters/base-stats/
  # uses 'Str' as a column name instead of 'S/M' like everywhere else.
  df = df.rename(columns={'Str': 'S/M'})

  # Drop unnecessary columns.
  df = df[[c for c in df.columns if c not in IGNORE_COLUMNS]]

  # This table lists Gonzales's class as Bandit, but other tables use Brigand.
  df['Class'] = df['Class'].replace(CLASS_RENAME_MAP)

  # Convert FE6 fan translation names to NOA.
  df['Name'] = df['Name'].replace(FE6_NOA_BY_SERENESFOREST_NAME)

  if is_fe7:
    # Only applies for FE7; Eliwood and Hector are non-lord classes in FE6.
    for lord in ['Lyn', 'Eliwood', 'Hector']:
      df.loc[df['Name'] == lord, 'Class'] = f'Lord ({lord})'
    df.loc[df['Name'] == 'Merlinus', 'Class'] = 'Transporter (Tent)'

  for lord in ['Eirika', 'Ephraim']:
    df.loc[df['Name'] == lord, 'Class'] = f'Lord ({lord})'

  # Replace single quotation mark with apostrophe.
  df['Name'] = df['Name'].replace({"L’Arachel": "L'Arachel"})

  # Remove asterisk for Lyn's Story Wallace.
  # Can also remove the General entry from base stats, since it can
  # be inferred from promotion gains.
  df['Name'] = df['Name'].replace({'Wallace *': 'Wallace'})
  df = df[~((df['Name'] == 'Wallace') & (df['Class'] == 'General'))]

  # Replace cells that say "Same as Ninian" with Ninian's actual stat.
  nils_idx = (df['Name'] == 'Nils') & (df['Variant'] != "Lyn's Tale")
  ninian_idx = df['Name'] == 'Ninian'
  if nils_idx.any():
    for c in df.columns:
      if df.loc[nils_idx, c].squeeze() == 'Same as Ninian':
        df.loc[nils_idx, c] = df.loc[ninian_idx, c].squeeze()

  # With the "Same as Ninian" values removed, we can cast strings to numeric.
  for c in df.columns:
    if c not in ['Name', 'Variant', 'Class']:
      df[c] = df[c].astype(float)

  # See above note about Knoll's auto-level.
  knoll_idx = (df['Name'] == 'Knoll')
  if knoll_idx.any():
    for stat_name, stat in KNOLL_BASE_STAT_BY_NAME.items():
      df.loc[knoll_idx, stat_name] = stat

  # Gonzales starts at Level 11 on Elffin's route, but otherwise
  # has the same stats.
  gonzalez_idx = (df['Name'] == 'Gonzalez')
  if gonzalez_idx.any():
    larum_route = df.loc[gonzalez_idx].copy()
    elffin_route = larum_route.copy()
    larum_route['Variant'] = "Larum's route"
    elffin_route['Variant'] = "Elffin's route"
    elffin_route['Lv'] = 11
    df = pd.concat([df.loc[~gonzalez_idx], larum_route, elffin_route])

  # Hugh's base stats decline by 1 for every time you decline his offer.
  # https://fireemblemwiki.org/wiki/Hugh#Starting_stats_and_growth_rates
  hugh_idx = (df['Name'] == 'Hugh')
  if hugh_idx.any():
    hugh_row_orig = df.loc[hugh_idx].copy()
    hugh_row_by_decline_times = {n: hugh_row_orig.copy() for n in range(4)}
    for n, hugh_row in hugh_row_by_decline_times.items():
      for c in ['HP', 'Skl', 'Spd', 'S/M', 'Lck', 'Def', 'Res']:
        hugh_row[c] -= n
      gold = {0: 10, 1: 8, 2: 6, 3: 5}[n]
      hugh_row['Variant'] = f'Pay {gold}k gold'
    df = pd.concat(
        [df.loc[~hugh_idx], *hugh_row_by_decline_times.values()])

  # Remove Hard Mode rows.
  df = df[~df['Name'].str.endswith(' HM')].copy()
  df = df[~df['Name'].str.endswith(' (HM)')].copy()
  # Add Hard Mode bonus levels.
  df = append_hm_bonus(df)

  # Add a new column for the character's gender.
  df = append_char_gender(df).set_index(
      ['Name', 'Variant', 'Class', 'Gender', 'HM Bonus']
  )

  return df


def process_char_growth_rates(df, is_fe6):
  df = df.copy()
  # Replace single quotation mark with apostrophe.
  df['Name'] = df['Name'].replace({"L’Arachel": "L'Arachel"})
  df['Name'] = df['Name'].replace(FE6_NOA_BY_SERENESFOREST_NAME)
  # Split "Nils/Ninian" row into two separate rows.
  nils = df.loc[df['Name'] == 'Nils/Ninian'].copy().assign(Name='Nils')
  ninian = nils.copy().assign(Name='Ninian')
  df = pd.concat([df[df['Name'] != 'Nils/Ninian'], nils, ninian])
  df = df.set_index(['Name'])
  # Cast numerical values to float.
  for c in df.columns:
    df[c] = df[c].astype(float)
  if is_fe6:
    # Extra characters have base stats on serenesforest, but not growth rates.
    # https://serenesforest.net/binding-blade/characters/base-stats/
    # https://serenesforest.net/binding-blade/characters/growth-rates/
    # Manually enter them from
    # https://fireemblem.fandom.com/wiki/Brunnya#Growth_Rates
    # or
    # https://fireemblemwiki.org/wiki/Brunnya#Stats
    extra_df = pd.DataFrame(FE6_EXTRA_CHARACTER_GROWTH_RATES).set_index('Name')
    df = pd.concat([df, extra_df])
  return df


def append_class_gender(df):
  df = df.copy()
  # Default gender to male, then override.
  df['Gender'] = 'M'
  # Set gender for female-locked classes.
  df.loc[df['Class'].isin(FEMALE_LOCKED_CLASSES), 'Gender'] = 'F'
  # Set gender for female variants of other classes.
  df.loc[df['Class'].str.endswith(' (F)'), 'Gender'] = 'F'
  # Remove gender suffix from class name.
  df['Class'] = df['Class'].apply(
      lambda x: x.removesuffix(' (M)').removesuffix(' (F)'))
  if 'Promotion' in df.columns:
    df['Promotion'] = df['Promotion'].apply(
        lambda x: x.removesuffix(' (M)').removesuffix(' (F)'))
  return df


def process_class_df(df):
  df = df.copy()
  # https://serenesforest.net/blazing-sword/classes/growth-rates/ has a
  # column named "Name" when it should be "Class".
  # Also clean up typos/inconsistencies in class names.
  df = df.rename(columns={'Name': 'Class'})

  df['Class'] = df['Class'].replace(CLASS_RENAME_MAP)
  # Add a new column for the class's gender.
  df = append_class_gender(df).set_index(['Class', 'Gender'])
  if 'Promotion' in df.columns:
    df['Promotion'] = df['Promotion'].replace(CLASS_RENAME_MAP)
    df = df.set_index('Promotion', append=True)

  # https://serenesforest.net/the-sacred-stones/classes/growth-rates/
  # lists all versions of the trainee classes under one row.
  # Split them into 3 rows.
  trainee_rows = df.index.get_level_values('Class').str.endswith(' (all)')
  if trainee_rows.any():
    remainder_df = df.loc[~trainee_rows]
    trainee_df_list = [
        pd.concat({
            x: df.loc[f'{trainee_class} (all)']
            for x in (trainee_class, f'{trainee_class} (2)', f'{trainee_class} (3)')
        }, names=['Class'])
        for trainee_class in ('Journeyman', 'Pupil', 'Recruit')
    ]
    df = pd.concat([remainder_df, *trainee_df_list])

  # Drop unnecessary columns.
  df = df[[c for c in df.columns if c not in IGNORE_COLUMNS]]

  # Cast numerical values to float.
  for c in df.columns:
    df[c] = df[c].astype(float)
  return df


def process_fe7_class_promo_gains(df):
  df = df.copy()
  indices = df.index.names
  df = df.reset_index()
  df = df.rename(columns={'Class': 'Promotion'}).copy()
  inverse = {v: k for k, v in FE7_PROMOTED_CLASS_BY_CLASS.items()}

  def get_previous_class(promotion, gender):
    if promotion == 'Bishop':
      return 'Monk' if gender == 'M' else 'Cleric'
    return inverse[promotion]
  df['Class'] = df.apply(lambda row: get_previous_class(
      row['Promotion'], row['Gender']), axis=1)
  transporter_row = pd.DataFrame(
      [{
          'Class': 'Transporter (Tent)',
          'Promotion': 'Transporter (Wagon)',
          'Gender': 'M',
          **{stat: 0 for stat in ('HP', 'S/M','Skl', 'Spd', 'Def', 'Res', 'Mov', 'Con')}
      }]
  )
  df = pd.concat([df, transporter_row], ignore_index=True)
  return df.set_index(['Class', 'Gender', 'Promotion'])


def process_fe6_class_max_stats(df):
  # https://serenesforest.net/binding-blade/classes/maximum-stats/
  # does not explicitly list max stats for HP, Lck, Mov because they are
  # constant for all classes.
  df = df.copy()
  df['HP'] = 60
  df['Lck'] = 30
  df['Mov'] = 15
  return df


In [5]:
# @title Read in tables from serenesforest.

def get_df_by_name(url_by_table_name, is_fe6, is_fe7, is_fe8):

  # Read in each table
  df_by_name = {
      table_name: get_tables_from_url(url)[0]
      for table_name, url in url_by_table_name.items()
      if table_name != 'char_base_stats'
  }
  if is_fe7:
    # Concat tables for Lyn's story and Eliwood/Hector's story.
    df_by_name['char_base_stats'] = pd.concat(
        dict(zip(
            ("Lyn's Tale", "Eliwood/Hector's Tale"),
            get_tables_from_url(url_by_table_name['char_base_stats'])
        )),
        names=['Variant']
    ).reset_index(-1, drop=True).reset_index()
  elif is_fe8:
    df_by_name['char_base_stats'] = pd.concat(
        get_tables_from_url(url_by_table_name['char_base_stats'])
    ).assign(Variant='All')
  else:
    df_by_name['char_base_stats'] = get_tables_from_url(
        url_by_table_name['char_base_stats'])[0].assign(Variant='All')
  # More post-processing and cleanup.
  df_by_name = {k: remove_redundant_header_rows(
      v) for k, v in df_by_name.items()}
  df_by_name['char_base_stats'] = process_char_base_stats(
      df_by_name['char_base_stats'], is_fe7=is_fe7)
  df_by_name['char_growth_rates'] = process_char_growth_rates(
      df_by_name['char_growth_rates'], is_fe6=is_fe6)
  for c in df_by_name:
    if c.startswith('class_'):
      df_by_name[c] = process_class_df(df_by_name[c])

  if is_fe7:
    df_by_name['class_promo_gains'] = process_fe7_class_promo_gains(
        df_by_name['class_promo_gains'])

  if is_fe6:
    df_by_name['class_max_stats'] = process_fe6_class_max_stats(
        df_by_name['class_max_stats'])

  for k, v in df_by_name.items():
    v.columns = pd.MultiIndex.from_product(
        [[k], v.columns], names=['stat_type', 'stat'])
  return df_by_name


fe6_df_by_name = get_df_by_name(
    fe6_url_by_table_name, is_fe6=True, is_fe7=False, is_fe8=False)
fe7_df_by_name = get_df_by_name(
    fe7_url_by_table_name, is_fe6=False, is_fe7=True, is_fe8=False)
fe8_df_by_name = get_df_by_name(
    fe8_url_by_table_name, is_fe6=False, is_fe7=False, is_fe8=True)


https://serenesforest.net/binding-blade/classes/growth-rates/
https://serenesforest.net/binding-blade/classes/maximum-stats/
https://serenesforest.net/binding-blade/classes/promotion-gains/
https://serenesforest.net/binding-blade/characters/growth-rates/
https://serenesforest.net/binding-blade/characters/base-stats/
https://serenesforest.net/blazing-sword/classes/growth-rates/
https://serenesforest.net/blazing-sword/classes/maximum-stats/
https://serenesforest.net/blazing-sword/classes/promotion-gains/
https://serenesforest.net/blazing-sword/characters/growth-rates/
https://serenesforest.net/blazing-sword/characters/base-stats/
https://serenesforest.net/the-sacred-stones/classes/growth-rates/
https://serenesforest.net/the-sacred-stones/classes/maximum-stats/
https://serenesforest.net/the-sacred-stones/classes/promotion-gains/
https://serenesforest.net/the-sacred-stones/characters/growth-rates/
https://serenesforest.net/the-sacred-stones/characters/base-stats/


In [6]:
# @title Create one data frame with all the class-level stats.

def get_class_df(df_by_name, is_fe8):
  class_df = (
      df_by_name['class_growth_rates']
      .join(df_by_name['class_max_stats'], how='outer')
  )
  # For non-promoted classes, get max stats from the "non-promoted" rows.
  non_promoted_rows = class_df['class_max_stats'].isna().all(axis=1)
  if is_fe8:
    # FE8 has different max Constitution stats for non-promoted classes
    # depending on if it is mounted or not.
    mounted_rows = (
        class_df
        .index
        .get_level_values('Class')
        .isin(FE8_MOUNTED_NON_PROMOTED_CLASSES)
    )
    for c in class_df['class_max_stats'].columns:
      class_df.loc[non_promoted_rows & mounted_rows, ('class_max_stats', c)] = (
          class_df.loc[('Non-promoted (mounted)', 'M'),
                       ('class_max_stats', c)]
      )
      class_df.loc[non_promoted_rows & ~mounted_rows, ('class_max_stats', c)] = (
          class_df.loc[('Non-promoted (foot)', 'M'),
                       ('class_max_stats', c)]
      )
    # Rename Lord classes.
    index_names = class_df.index.names
    class_df = class_df.reset_index()
    class_df.loc[(class_df['Class']=='Lord') & (class_df['Gender'] == 'F'), 'Class'] = 'Lord (Eirika)'
    class_df.loc[(class_df['Class']=='Lord') & (class_df['Gender'] == 'M'), 'Class'] = 'Lord (Ephraim)'
    class_df = class_df.set_index(index_names)
  else:
    # Max stats are the same for all non-promoted classes in FE6 and FE7.
    for c in class_df['class_max_stats'].columns:
      class_df.loc[non_promoted_rows, ('class_max_stats', c)] = (
          class_df.loc[('Non-promoted', 'M'), ('class_max_stats', c)]
      )
  return class_df


fe6_class_df = get_class_df(fe6_df_by_name, is_fe8=False)
fe7_class_df = get_class_df(fe7_df_by_name, is_fe8=False)
fe8_class_df = get_class_df(fe8_df_by_name, is_fe8=True)
fe8_class_df[['class_growth_rates', 'class_max_stats']].tail(60)


stat_type                     class_growth_rates                          \
stat                                          HP   S/M   Skl   Spd   Lck   
Class                  Gender                                              
Maelduin               M                    75.0  30.0  30.0  18.0  10.0   
Mage                   F                    55.0  55.0  40.0  35.0  20.0   
                       M                    55.0  55.0  40.0  35.0  20.0   
Mage Knight            F                    45.0  40.0  30.0  40.0  40.0   
                       M                    45.0  40.0  30.0  40.0  30.0   
Manakete               F                    95.0  40.0  30.0  20.0  25.0   
                       M                    95.0  40.0  30.0  20.0  25.0   
Mauthedoog             M                    70.0  35.0  45.0  45.0  30.0   
Mercenary              F                    80.0  40.0  40.0  32.0  30.0   
                       M                    80.0  40.0  40.0  32.0  30.0   
Mogall                 M                    50.0  45.0  32.0  30.0  30.0   
Monk                   M                    50.0  30.0  35.0  32.0  45.0   
Myrmidon               F                    70.0  35.0  40.0  40.0  30.0   
                       M                    70.0  35.0  40.0  40.0  30.0   
Necromancer            M                    45.0  55.0  30.0  25.0  20.0   
Non-promoted (foot)    M                     NaN   NaN   NaN   NaN   NaN   
Non-promoted (mounted) M                     NaN   NaN   NaN   NaN   NaN   
Paladin                F                    70.0  25.0  35.0  25.0  25.0   
                       M                    70.0  25.0  30.0  18.0  25.0   
Pegasus Knight         F                    65.0  35.0  40.0  40.0  35.0   
Phantom                M                     0.0  55.0  35.0  45.0  50.0   
Pirate                 M                    75.0  50.0  35.0  25.0  15.0   
Pontifex               M                    10.0   0.0   0.0   0.0   0.0   
Priest                 M                    50.0  30.0  35.0  32.0  45.0   
Pupil                  M                    55.0  55.0  40.0  35.0  40.0   
Pupil (2)              M                    55.0  55.0  40.0  35.0  40.0   
Pupil (3)              M                    55.0  55.0  40.0  35.0  40.0   
Ranger                 F                    60.0  25.0  30.0  35.0  25.0   
                       M                    60.0  25.0  30.0  35.0  25.0   
Recruit                F                    75.0  45.0  40.0  40.0  40.0   
Recruit (2)            F                    75.0  45.0  40.0  40.0  40.0   
Recruit (3)            F                    75.0  45.0  40.0  40.0  40.0   
Revenant               M                    95.0  50.0  30.0  20.0  10.0   
Rogue                  M                    50.0  10.0  45.0  35.0  40.0   
Sage                   F                    45.0  45.0  30.0  25.0  15.0   
                       M                    45.0  45.0  30.0  25.0  15.0   
Shaman                 F                    50.0  45.0  32.0  30.0  20.0   
                       M                    50.0  50.0  32.0  30.0  20.0   
Sniper                 F                    65.0  30.0  30.0  20.0  30.0   
                       M                    65.0  30.0  30.0  20.0  30.0   
Soldier                M                    80.0  50.0  30.0  20.0  25.0   
Summoner               F                    45.0  50.0  30.0  25.0  20.0   
                       M                    45.0  50.0  30.0  25.0  20.0   
Swordmaster            F                    65.0  25.0  30.0  30.0  25.0   
                       M                    65.0  25.0  30.0  30.0  25.0   
Tarvos                 M                    80.0  35.0  40.0  28.0  10.0   
Thief                  M                    50.0   5.0  45.0  40.0  40.0   
Troubadour             F                    50.0  25.0  35.0  55.0  45.0   
Valkyrie               F                    45.0  35.0  25.0  45.0  40.0   
Warrior                M                    80.0

In [7]:
# @title Create one data frame with all the char-level stats.


def get_char_df(df_by_name):

  char_df = df_by_name['char_base_stats'].join(
      df_by_name['char_growth_rates'], how='outer')
  # Promotion to second class`
  promo2 = df_by_name['class_promo_gains'].copy()
  promo2.index = promo2.index.rename('SecondClass', level='Promotion')
  promo2.columns = promo2.columns.set_levels(
      ['second_class_promo_gains'], level='stat_type')
  char_df = char_df.join(promo2, how='left')
  # Promotion to third class
  promo3 = df_by_name['class_promo_gains'].copy()
  promo3.index = promo3.index.rename(
      'ThirdClass', level='Promotion').rename('SecondClass', level='Class')
  promo3.columns = promo3.columns.set_levels(
      ['third_class_promo_gains'], level='stat_type')
  char_df = char_df.join(promo3, how='left')
  # char_df = append_second_class(char_df, promoted_classes_by_class)
  return char_df


fe6_char_df = get_char_df(fe6_df_by_name)
fe7_char_df = get_char_df(fe7_df_by_name)
fe8_char_df = get_char_df(fe8_df_by_name)
fe6_char_df


stat_type                                                                                char_base_stats  \
stat                                                                                                  Lv   
Name   Variant                     Class          Gender HM Bonus SecondClass ThirdClass                   
Roy    All                         Lord           M      0        Master Lord NaN                    1.0   
Marcus All                         Paladin        M      0        NaN         NaN                    1.0   
Alen   All                         Cavalier       M      0        Paladin     NaN                    1.0   
Lance  All                         Cavalier       M      0        Paladin     NaN                    1.0   
Wolt   All                         Archer         M      0        Sniper      NaN                    1.0   
...                                                                                                  ...   
Cath   Chapter 22                  Thief          F      14       NaN         NaN                    5.0   
Klein  Chapter 11A (Larum's route) Sniper         M      9        NaN         NaN                    1.0   
Thea   Chapter 11A (Larum's route) Pegasus Knight F      9        Falcoknight NaN                    8.0   
Klein  Chapter 10B Elffin's route) Sniper         M      8        NaN         NaN                    1.0   
Thea   Chapter 10B Elffin's route) Pegasus Knight F      8        Falcoknight NaN                    8.0   

stat_type                                                                                       \
stat                                                                                        HP   
Name   Variant                     Class          Gender HM Bonus SecondClass ThirdClass         
Roy    All                         Lord           M      0        Master Lord NaN         18.0   
Marcus All                         Paladin        M      0        NaN         NaN         32.0   
Alen   All                         Cavalier       M      0        Paladin     NaN         21.0   
Lance  All                         Cavalier       M      0        Paladin     NaN         20.0   
Wolt   All                         Archer         M      0        Sniper      NaN         18.0   
...                                                                                        ...   
Cath   Chapter 22                  Thief          F      14       NaN         NaN         16.0   
Klein  Chapter 11A (Larum's route) Sniper         M      9        NaN         NaN         27.0   
Thea   Chapter 11A (Larum's route) Pegasus Knight F      9        Falcoknight NaN         22.0   
Klein  Chapter 10B Elffin's route) Sniper         M      8        NaN         NaN         27.0   
Thea   Chapter 10B Elffin's route) Pegasus Knight F      8        Falcoknight NaN         22.0   

stat_type                                                                                       \
stat                                                                                       S/M   
Name   Variant                     Class          Gender HM Bonus SecondClass ThirdClass         
Roy    All                         Lord           M      0        Master Lord NaN          5.0   
Marcus All                         Paladin        M      0        NaN         NaN          9.0   
Alen   All                         Cavalier       M      0        Paladin     NaN          7.0   
Lance  All                         Cavalier       M      0        Paladin     NaN          5.0   
Wolt   All                         Archer         M      0        Sniper      NaN          4.0   
...                                                                                        ...   
Cath   Chapter 22                  Thief          F      14       NaN         NaN          3.0   
Klein  Chapter 11A (Larum's route) Sniper         M      9        NaN         NaN         13.0   
Thea   Chapter 11A (Larum's route) P

In [8]:
def get_final_df(char_df, class_df):
  def get_class_df_copy(new_class_name, class_prefix):
    new_class_df = class_df.copy()
    new_class_df.index.names = [new_class_name, 'Gender']
    new_class_df.columns = pd.MultiIndex.from_tuples(
        [(f'{class_prefix}_{df_name}', stat)
        for df_name, stat in new_class_df.columns],
        names=new_class_df.columns.names
    )
    return new_class_df

  second_class_df = get_class_df_copy('SecondClass', 'second')
  third_class_df = get_class_df_copy('ThirdClass', 'third')

  final_df = char_df.join(class_df).join(
      second_class_df).join(third_class_df)
  return final_df


fe6_final_df = get_final_df(fe6_char_df, fe6_class_df)
fe7_final_df = get_final_df(fe7_char_df, fe7_class_df)
fe8_final_df = get_final_df(fe8_char_df, fe8_class_df)


In [9]:
# @title Rerolls to avoid empty level ups
# @markdown In the GBA games, if no stats increase during a level up,
# @markdown the game rerolls the level up.
# @markdown If again no stats increase, it rerolls once final time.
# @markdown Because of the two extra opportunities to get stat increases,
# @markdown the effective growth rate is slightly higher
# @markdown than the [original] growth rate, with a bigger impact for
# @markdown characters with very low [original] growth rates (Niime, Yoder).

# https://forums.serenesforest.net/topic/91108-a-deep-dive-into-level-up-mechanics/
# https://fireemblem.fandom.com/wiki/Level#Leveling_Up_and_Stat_Growth

def get_adjusted_growth_rates(row):
  growth_rates = row.xs('char_growth_rates')
  if (growth_rates > 100).any():
    # If any growth rate is larger than 100, the stat is guaranteed to increase
    # so no rerolls are necessary.
    return growth_rates
  else:
    reroll_prob = (1.0 - growth_rates / 100.0).prod()
    growth_rates /= 100.0
    adjusted_growth_rates = (
        # Stat increases on the first try.
        growth_rates
        # Empty level up on the first try, then increases on the second try.
        + reroll_prob * growth_rates
        # Empty level ups on the first two tries, then increases on the third try.
        + reroll_prob**2 * growth_rates
    )
    return 100.0 * adjusted_growth_rates

final_df = pd.concat(
  {'FE6': fe6_final_df, 'FE7': fe7_final_df, 'FE8': fe8_final_df},
  names=['Game']
)
final_df.index = pd.MultiIndex.from_frame(
        final_df.index.to_frame().fillna('N/A')
)
adjusted_growth_rates = final_df.apply(get_adjusted_growth_rates, axis=1)
adjusted_growth_rates.columns = pd.MultiIndex.from_product([['char_growth_rates_adjusted'], adjusted_growth_rates.columns], names=['stat_type', 'stat'])
final_df = pd.concat([final_df, adjusted_growth_rates], axis=1)
final_df

stat_type                                                                char_base_stats  \
stat                                                                                  Lv   
Game Name   Variant Class         Gender HM Bonus SecondClass ThirdClass                   
FE6  Roy    All     Lord          M      0        Master Lord N/A                    1.0   
     Marcus All     Paladin       M      0        N/A         N/A                    1.0   
     Alen   All     Cavalier      M      0        Paladin     N/A                    1.0   
     Lance  All     Cavalier      M      0        Paladin     N/A                    1.0   
     Wolt   All     Archer        M      0        Sniper      N/A                    1.0   
...                                                                                  ...   
FE8  Glen   All     Wyvern Lord   M      0        N/A         N/A                   12.0   
     Hayden All     Ranger        M      0        N/A         N/A                   10.0   
     Valter All     Wyvern Knight M      0        N/A         N/A                   13.0   
     Fado   All     General       M      0        N/A         N/A                   11.0   
     Lyon   All     Necromancer   M      0        N/A         N/A                   14.0   

stat_type                                                                       \
stat                                                                        HP   
Game Name   Variant Class         Gender HM Bonus SecondClass ThirdClass         
FE6  Roy    All     Lord          M      0        Master Lord N/A         18.0   
     Marcus All     Paladin       M      0        N/A         N/A         32.0   
     Alen   All     Cavalier      M      0        Paladin     N/A         21.0   
     Lance  All     Cavalier      M      0        Paladin     N/A         20.0   
     Wolt   All     Archer        M      0        Sniper      N/A         18.0   
...                                                                        ...   
FE8  Glen   All     Wyvern Lord   M      0        N/A         N/A         46.0   
     Hayden All     Ranger        M      0        N/A         N/A         37.0   
     Valter All     Wyvern Knight M      0        N/A         N/A         45.0   
     Fado   All     General       M      0        N/A         N/A         46.0   
     Lyon   All     Necromancer   M      0        N/A         N/A         44.0   

stat_type                                                                       \
stat                                                                       S/M   
Game Name   Variant Class         Gender HM Bonus SecondClass ThirdClass         
FE6  Roy    All     Lord          M      0        Master Lord N/A          5.0   
     Marcus All     Paladin       M      0        N/A         N/A          9.0   
     Alen   All     Cavalier      M      0        Paladin     N/A          7.0   
     Lance  All     Cavalier      M      0        Paladin     N/A          5.0   
     Wolt   All     Archer        M      0        Sniper      N/A          4.0   
...                                                                        ...   
FE8  Glen   All     Wyvern Lord   M      0        N/A         N/A         20.0   
     Hayden All     Ranger        M      0        N/A         N/A         17.0   
     Valter All     Wyvern Knight M      0        N/A         N/A         19.0   
     Fado   All     General       M      0        N/A         N/A         20.0   
     Lyon   All     Necromancer   M      0        N/A         N/A         22.0   

stat_type                                                                       \
stat                                                                       Skl   
Game Name   Variant Class         Gender HM Bonus SecondClass ThirdClass         
FE6  Roy    All     Lord          M      0        Master Lord N/A          5.0   
     Marcus All     Paladin       M      0        N/A         N/A         14.0   
   

In [10]:
final_df.query('Game == "FE6"').tail(50)

stat_type                                                                                             char_base_stats  \
stat                                                                                                               Lv   
Game Name      Variant                     Class           Gender HM Bonus SecondClass     ThirdClass                   
FE6  Astolfo   All                         Thief           M      0        N/A             N/A                   10.0   
     Lilina    All                         Mage            F      0        Sage            N/A                    1.0   
     Gwendolyn All                         Knight          F      0        General         N/A                    1.0   
     Barthe    All                         Knight          M      0        General         N/A                    9.0   
     Ogier     All                         Mercenary       M      0        Hero            N/A                    3.0   
     Fir       All                         Myrmidon        F      8        Swordmaster     N/A                    1.0   
     Sin       All                         Nomad           M      8        Nomadic Trooper N/A                    5.0   
     Geese     All                         Pirate          M      0        Berserker       N/A                   10.0   
     Larum     All                         Dancer          F      0        N/A             N/A                    1.0   
     Echidna   All                         Hero            F      0        N/A             N/A                    1.0   
     Elffin    All                         Bard            M      0        N/A             N/A                    1.0   
     Bartre    All                         Warrior         M      0        N/A             N/A                    2.0   
     Raigh     All                         Shaman          M      0        Druid           N/A                   12.0   
     Melady    All                         Wyvern Rider    F      10       Wyvern Lord     N/A                   10.0   
     Cecilia   All                         Valkyrie        F      0        N/A             N/A                    1.0   
     Sophia    All                         Shaman          F      0        Druid           N/A                    1.0   
     Igrene    All                         Sniper          F      0        N/A             N/A                    1.0   
     Garret    All                         Berserker       M      11       N/A             N/A                    1.0   
     Fae       All                         Manakete        F      0        N/A             N/A                    1.0   
     Zeiss     All                         Wyvern Rider    M      11       Wyvern Lord     N/A                    7.0   
     Douglas   All                         General         M      0        N/A             N/A                    8.0   
     Niime     All                         Druid           F      0        N/A             N/A                   18.0   
     Dayan     All                         Nomadic Trooper M      0        N/A             N/A                   12.0   
     Juno      All                         Falcoknight     F      0        N/A             N/A                    9.0   
     Yoder     All                         Bishop          M      0        N/A             N/A                   20.0   
     Karel     All                         Swordmaster     M      0        N/A             N/A                   19.0   
     Narcian   All                         Wyvern Lord     M      0        N/A             N/A                   10.0   
     Galle     All                         Wyvern Lord     M      0        N/A             N/A                   18.0   
     Hector    All                         General         M      0        N/A             N/A                   20.0   
     Brunnya   All                         Sage            F      0        N/A             N/A                   20.0   
     El

In [11]:
final_df.query('Name=="Gonzalez"').stack('stat_type')

/tmp/ipykernel_11213/2806678157.py:1: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  final_df.query('Name=="Gonzalez"').stack('stat_type')


stat                                                                                                      Lv  \
Game Name     Variant        Class   Gender HM Bonus SecondClass ThirdClass stat_type                          
FE6  Gonzalez Larum's route  Brigand M      8        Berserker   N/A        char_base_stats              5.0   
                                                                            char_growth_rates            NaN   
                                                                            char_growth_rates_adjusted   NaN   
                                                                            class_growth_rates           NaN   
                                                                            class_max_stats              NaN   
                                                                            second_class_growth_rates    NaN   
                                                                            second_class_max_stats       NaN   
                                                                            second_class_promo_gains     NaN   
              Elffin's route Brigand M      8        Berserker   N/A        char_base_stats             11.0   
                                                                            char_growth_rates            NaN   
                                                                            char_growth_rates_adjusted   NaN   
                                                                            class_growth_rates           NaN   
                                                                            class_max_stats              NaN   
                                                                            second_class_growth_rates    NaN   
                                                                            second_class_max_stats       NaN   
                                                                            second_class_promo_gains     NaN   

stat                                                                                                          HP  \
Game Name     Variant        Class   Gender HM Bonus SecondClass ThirdClass stat_type                              
FE6  Gonzalez Larum's route  Brigand M      8        Berserker   N/A        char_base_stats             36.00000   
                                                                            char_growth_rates           90.00000   
                                                                            char_growth_rates_adjusted  90.71416   
                                                                            class_growth_rates          82.00000   
                                                                            class_max_stats             60.00000   
                                                                            second_class_growth_rates   58.00000   
                                                                            second_class_max_stats      60.00000   
                                                                            second_class_promo_gains     4.00000   
              Elffin's route Brigand M      8        Berserker   N/A        char_base_stats             36.00000   
                                                                            char_growth_rates           90.00000   
                                                                            char_growth_rates_adjusted  90.71416   
                                                                            class_growth_rates          82.00000   
                                                                            class_max_stats             60.00000   
                                                                            second_class_growth_rates   58.00000   
                                                                            second_class_max_stats      60.00000   
           

In [12]:
final_df[final_df.index.get_level_values('ThirdClass') != 'N/A']

stat_type                                                                    char_base_stats  \
stat                                                                                      Lv   
Game Name   Variant Class      Gender HM Bonus SecondClass    ThirdClass                       
FE8  Ross   All     Journeyman M      0        Fighter        Warrior                    1.0   
                                                              Hero                       1.0   
                                               Pirate         Berserker                  1.0   
                                                              Warrior                    1.0   
                                               Journeyman (2) Hero                       1.0   
                                                              Journeyman (3)             1.0   
     Amelia All     Recruit    F      0        Cavalier       Paladin                    1.0   
                                                              Great Knight               1.0   
                                               Knight         General                    1.0   
                                                              Great Knight               1.0   
                                               Recruit (2)    Paladin                    1.0   
                                                              Recruit (3)                1.0   
     Ewan   All     Pupil      M      0        Mage           Sage                       1.0   
                                                              Mage Knight                1.0   
                                               Shaman         Druid                      1.0   
                                                              Summoner                   1.0   
                                               Pupil (2)      Sage                       1.0   
                                                              Pupil (3)                  1.0   

stat_type                                                                           \
stat                                                                            HP   
Game Name   Variant Class      Gender HM Bonus SecondClass    ThirdClass             
FE8  Ross   All     Journeyman M      0        Fighter        Warrior         15.0   
                                                              Hero            15.0   
                                               Pirate         Berserker       15.0   
                                                              Warrior         15.0   
                                               Journeyman (2) Hero            15.0   
                                                              Journeyman (3)  15.0   
     Amelia All     Recruit    F      0        Cavalier       Paladin         16.0   
                                                              Great Knight    16.0   
                                               Knight         General         16.0   
                                                              Great Knight    16.0   
                                               Recruit (2)    Paladin         16.0   
                                                              Recruit (3)     16.0   
     Ewan   All     Pupil      M      0        Mage           Sage            15.0   
                                                              Mage Knight     15.0   
                                               Shaman         Druid           15.0   
                                                              Summoner        15.0   
                                               Pupil (2)      Sage            15.0   
                                                              Pupil (3)       15.0   

stat_type                                                                          \
stat                                                                          S/M   
Game N

In [13]:
final_df.query('Name=="Melady"').stack('stat_type', future_stack=True).round(3)

stat                                                                                                  Lv  \
Game Name   Variant Class        Gender HM Bonus SecondClass ThirdClass stat_type                          
FE6  Melady All     Wyvern Rider F      10       Wyvern Lord N/A        char_base_stats             10.0   
                                                                        char_growth_rates            NaN   
                                                                        second_class_promo_gains     NaN   
                                                                        third_class_promo_gains      NaN   
                                                                        class_growth_rates           NaN   
                                                                        class_max_stats              NaN   
                                                                        second_class_growth_rates    NaN   
                                                                        second_class_max_stats       NaN   
                                                                        third_class_growth_rates     NaN   
                                                                        third_class_max_stats        NaN   
                                                                        char_growth_rates_adjusted   NaN   

stat                                                                                                    HP  \
Game Name   Variant Class        Gender HM Bonus SecondClass ThirdClass stat_type                            
FE6  Melady All     Wyvern Rider F      10       Wyvern Lord N/A        char_base_stats             30.000   
                                                                        char_growth_rates           75.000   
                                                                        second_class_promo_gains     5.000   
                                                                        third_class_promo_gains        NaN   
                                                                        class_growth_rates          80.000   
                                                                        class_max_stats             60.000   
                                                                        second_class_growth_rates   75.000   
                                                                        second_class_max_stats      60.000   
                                                                        third_class_growth_rates       NaN   
                                                                        third_class_max_stats          NaN   
                                                                        char_growth_rates_adjusted  76.498   

stat                                                                                                   S/M  \
Game Name   Variant Class        Gender HM Bonus SecondClass ThirdClass stat_type                            
FE6  Melady All     Wyvern Rider F      10       Wyvern Lord N/A        char_base_stats             12.000   
                                                                        char_growth_rates           50.000   
                                                                        second_class_promo_gains     2.000   
                                                                        third_class_promo_gains        NaN   
                                                                        class_growth_rates          45.000   
                                                                        class_max_stats             20.000   
                                                                        second_class_growth_rates   40.000   
                                                                        second_class_max_stats      25.000   
                                                                

In [14]:
# @title FE7 Marcus's original and effective growth rates
row = final_df.query('Name=="Marcus" & Game=="FE7"').squeeze()
row.unstack('stat').query('stat_type in ("char_growth_rates", "char_growth_rates_adjusted")').round(3)

stat,Con,Def,HP,Lck,Lv,Mov,Res,S/M,Skl,Spd
stat_type,,,,,,,,,,
char_growth_rates,NaN,15.000,65.000,30.000,NaN,NaN,35.000,30.000,50.00,25.00
char_growth_rates_adjusted,NaN,15.552,67.392,31.104,NaN,NaN,36.288,31.104,51.84,25.92


In [15]:
# @title Niime's original and effective growth rates
row = final_df.query('Name=="Niime"').squeeze()
row.unstack('stat').query('stat_type in ("char_growth_rates", "char_growth_rates_adjusted")').round(3)

stat,Con,Def,HP,Lck,Lv,Mov,Res,S/M,Skl,Spd
stat_type,,,,,,,,,,
char_growth_rates,NaN,15.000,25.000,5.00,NaN,NaN,20.000,15.000,15.000,15.000
char_growth_rates_adjusted,NaN,20.791,34.652,6.93,NaN,NaN,27.722,20.791,20.791,20.791


In [16]:
# @title Yoder's original and effective growth rates
row = final_df.query('Name=="Yoder"').squeeze()
row.unstack('stat').query('stat_type in ("char_growth_rates", "char_growth_rates_adjusted")').round(3)

stat,Con,Def,HP,Lck,Lv,Mov,Res,S/M,Skl,Spd
stat_type,,,,,,,,,,
char_growth_rates,NaN,10.000,20.000,20.000,NaN,NaN,20.000,30.000,15.000,10.000
char_growth_rates_adjusted,NaN,13.076,26.153,26.153,NaN,NaN,26.153,39.229,19.615,13.076


In [17]:
row = final_df.query('Name=="Melady"').squeeze()
row.unstack('stat').query('stat_type in ("char_growth_rates", "char_growth_rates_adjusted")').round(3)

stat,Con,Def,HP,Lck,Lv,Mov,Res,S/M,Skl,Spd
stat_type,,,,,,,,,,
char_growth_rates,NaN,20.0,75.000,25.000,NaN,NaN,5.0,50.000,50.000,45.000
char_growth_rates_adjusted,NaN,20.4,76.498,25.499,NaN,NaN,5.1,50.999,50.999,45.899


In [18]:

drive.mount('/content/drive', force_remount=True)

final_df.stack('stat_type', future_stack=True).to_csv('/content/drive/MyDrive/fe_gba_stats/final.csv')


MessageError: Error: credential propagation was unsuccessful

In [ ]:
# pd.read_csv('/content/drive/MyDrive/fe_gba_stats/final.csv', keep_default_na=False)

In [ ]:
final_flat = final_df.copy()
final_flat.columns = final_flat.columns.get_level_values(0) + '_' + final_flat.columns.get_level_values(1)

drive.mount('/content/drive', force_remount=True)

final_flat.to_csv('/content/drive/MyDrive/fe_gba_stats/final_flat.csv')
